In [1]:
import os
# Enforce single GPU execution to prevent multi-GPU conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import login, hf_hub_download

# ==============================================================================
# GLOBAL DEBUG CONTROL
# Set DEBUG = True for fast pipeline verification, False for full production run.
# ==============================================================================
DEBUG = False

DEBUG_CONFIG = {
    "train_samples": 64,
    "dev_samples": 16,
    "test_samples": 8,
    "epochs": 3,
    "batch_size": 2,
    "log_interval": 2,
    "lr": 3e-4,
    "lora_rank": 16,
    "lora_alpha": 32,
    "max_doc_len": 768,
    "max_summ_len": 256,
    "gen_max_new_tokens": 128
}

FULL_CONFIG = {
    "train_samples": None,  # Full dataset
    "dev_samples": None,
    "test_samples": None,
    "epochs": 3,
    "batch_size": 2,
    "log_interval": 20,
    "lr": 3e-4,
    "lora_rank": 16,
    "lora_alpha": 32,
    "max_doc_len": 1536,
    "max_summ_len": 384,
    "gen_max_new_tokens": 256
}

CFG = DEBUG_CONFIG if DEBUG else FULL_CONFIG

print(f"=== Phase 6 Execution Mode: {'DEBUG (Fast Verification Run)' if DEBUG else 'PRODUCTION RUN'} ===")
print(f"Configuration: {CFG}\n")

=== Phase 6 Execution Mode: PRODUCTION RUN ===
Configuration: {'train_samples': None, 'dev_samples': None, 'test_samples': None, 'epochs': 3, 'batch_size': 2, 'log_interval': 20, 'lr': 0.0003, 'lora_rank': 16, 'lora_alpha': 32, 'max_doc_len': 1536, 'max_summ_len': 384, 'gen_max_new_tokens': 256}



In [ ]:
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO_HYBRID = "A-240242/legal-hybridencoder-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_summ_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_6_Summ/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 6 Abstractive Summarization Data...")
full_train_data = load_summ_file("Train/train.json")
test_data = load_summ_file("Test/test.json")

print(f"Loaded {len(full_train_data)} total training examples and {len(test_data)} test examples.")

# Construct a held-out validation split (10%) from train.json
np.random.seed(42)
indices = np.arange(len(full_train_data))
np.random.shuffle(indices)

val_size = max(1, int(0.10 * len(full_train_data)))
val_indices = set(indices[:val_size])

train_data = [full_train_data[i] for i in range(len(full_train_data)) if i not in val_indices]
dev_data = [full_train_data[i] for i in val_indices]

print(f"Split data into {len(train_data)} training samples and {len(dev_data)} validation samples.")

if DEBUG and CFG["train_samples"]:
    train_data = train_data[:CFG["train_samples"]]
    dev_data = dev_data[:CFG["dev_samples"]]
    test_data = test_data[:CFG["test_samples"]]
    print(f"\n[DEBUG] Subsampled train to {len(train_data)}, dev to {len(dev_data)}, and test to {len(test_data)} records.")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Loaded 7030 total training examples and 100 test examples.
Split data into 6327 training samples and 703 validation samples.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class LegalSummarizationDataset(Dataset):
    """
    Causal Language Modeling Dataset for Abstractive Summarization.
    Prompt Format: "Document:\n{document}\n\nSummary:\n{summary}"
    Labels for document tokens are masked with -100 so loss is computed strictly on summary tokens.
    """
    def __init__(self, raw_records, tokenizer, max_doc_len=1536, max_summ_len=384, is_test=False):
        self.samples = []
        self.is_test = is_test
        
        for record in raw_records:
            doc_id = record.get("id")
            doc_text = record.get("document", "")
            
            if is_test:
                prompt = f"Document:\n{doc_text}\n\nSummary:\n"
                prompt_ids = tokenizer.encode(prompt, truncation=True, max_length=max_doc_len)
                self.samples.append({
                    "id": doc_id,
                    "input_ids": prompt_ids,
                    "prompt_text": prompt
                })
            else:
                summ_text = record.get("summary", "")
                
                prompt_prefix = f"Document:\n{doc_text}\n\nSummary:\n"
                prefix_ids = tokenizer.encode(prompt_prefix, truncation=True, max_length=max_doc_len)
                summ_ids = tokenizer.encode(summ_text, truncation=True, max_length=max_summ_len)
                
                if tokenizer.eos_token_id not in summ_ids:
                    summ_ids.append(tokenizer.eos_token_id)
                    
                full_input_ids = prefix_ids + summ_ids
                labels = [-100] * len(prefix_ids) + summ_ids
                
                self.samples.append({
                    "id": doc_id,
                    "input_ids": full_input_ids,
                    "labels": labels
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def summ_collate_fn(batch):
    if "labels" in batch[0]:
        input_ids_list = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
        labels_list = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
        doc_ids = [item["id"] for item in batch]
        
        padded_input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id
        )
        padded_labels = torch.nn.utils.rnn.pad_sequence(
            labels_list, batch_first=True, padding_value=-100
        )
        attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()
        
        return {
            "id": doc_ids,
            "input_ids": padded_input_ids,
            "attention_mask": attention_mask,
            "labels": padded_labels
        }
    else:
        input_ids_list = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
        doc_ids = [item["id"] for item in batch]
        
        padded_input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id
        )
        attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()
        
        return {
            "id": doc_ids,
            "input_ids": padded_input_ids,
            "attention_mask": attention_mask
        }

print("Building Phase 6 DataLoaders...")
train_dataset = LegalSummarizationDataset(train_data, tokenizer, max_doc_len=CFG["max_doc_len"], max_summ_len=CFG["max_summ_len"])
dev_dataset = LegalSummarizationDataset(dev_data, tokenizer, max_doc_len=CFG["max_doc_len"], max_summ_len=CFG["max_summ_len"])

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, collate_fn=summ_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=CFG["batch_size"], shuffle=False, collate_fn=summ_collate_fn)

print(f"Created {len(train_dataset)} train samples and {len(dev_dataset)} dev samples.")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Building Phase 6 DataLoaders...
Created 6327 train samples and 703 dev samples.


In [4]:
class LayerNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(-1, keepdim=True)
        variance = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(variance + self.eps) * self.weight + self.bias


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int):
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    cos = cos[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    sin = sin[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class HybridCausalAttention(nn.Module):
    """Causal Multi-Head Self-Attention with RoPE."""
    def __init__(self, hidden_size=768, num_heads=12):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        B, S, _ = x.shape
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        if cos is not None and sin is not None:
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
            
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Causal triangular mask
        causal_mask = torch.full((S, S), float("-inf"), device=x.device)
        causal_mask = torch.triu(causal_mask, diagonal=1)
        scores = scores + causal_mask[None, None, :, :]
        
        if attention_mask is not None:
            padding_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0
            scores = scores + padding_mask
            
        probs = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.hidden_size)
        
        return self.o_proj(attn_out)


class HybridConvFFN(nn.Module):
    """
    Hybrid Feed-Forward Network combining 
    SwiGLU Linear Projections with 1D Depthwise Local Convolution.
    """
    def __init__(self, hidden_size=768, intermediate_size=2048, kernel_size=3):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=intermediate_size,
            out_channels=intermediate_size,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=intermediate_size,
            bias=False
        )
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gated = F.silu(self.gate_proj(x)) * self.up_proj(x)
        gated_conv = gated.transpose(1, 2)
        gated_conv = self.conv1d(gated_conv).transpose(1, 2)
        return self.down_proj(gated_conv)


class HybridDecoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = LayerNorm(hidden_size)
        self.self_attn = HybridCausalAttention(hidden_size, num_heads)
        self.post_attention_layernorm = LayerNorm(hidden_size)
        self.mlp = HybridConvFFN(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        residual = x
        x = self.input_layernorm(x)
        x = self.self_attn(x, attention_mask=attention_mask, cos=cos, sin=sin)
        x = residual + x
        
        residual = x
        x = self.post_attention_layernorm(x)
        x = self.mlp(x)
        x = residual + x
        return x


class HybridBackbone(nn.Module):
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            HybridDecoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm = LayerNorm(hidden_size)
        self.rotary_emb = RotaryEmbedding(hidden_size // num_heads)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        B, S = input_ids.shape
        x = self.embed_tokens(input_ids)
        cos, sin = self.rotary_emb(x, S)
        
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask, cos=cos, sin=sin)
            
        x = self.norm(x)
        return x


class CustomHybridEncoderCausalLM(nn.Module):
    """Custom Hybrid Encoder Causal Language Model."""
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.model = HybridBackbone(vocab_size, hidden_size, num_layers, num_heads, intermediate_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, labels: torch.Tensor = None):
        hidden_states = self.model(input_ids, attention_mask=attention_mask)
        logits = self.lm_head(hidden_states)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            
        return {"loss": loss, "logits": logits}

print("Custom Hybrid Encoder Causal LM Architecture written from scratch successfully.")

Custom Hybrid Encoder Causal LM Architecture written from scratch successfully.


In [5]:
class LoRAAdapter(nn.Module):
    """Low-Rank Adaptation Pair (A, B) in FP32."""
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        self.scaling = alpha / rank
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)  # Zero initialization preserves base outputs at epoch 0

    def forward(self, x):
        return self.scaling * self.lora_B(self.lora_A(x))


class LoRAWrappedLinear(nn.Module):
    """Wraps a frozen base Linear projection layer with a LoRA adapter."""
    def __init__(self, base_linear: nn.Linear, rank=16, alpha=32):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
            
        self.adapter = LoRAAdapter(
            base_linear.in_features, base_linear.out_features, rank=rank, alpha=alpha
        )

    def forward(self, x):
        return self.base(x) + self.adapter(x)


# 1. Instantiate Custom Hybrid Encoder Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomHybridEncoderCausalLM(
    vocab_size=len(tokenizer), 
    hidden_size=768, 
    num_layers=6, 
    num_heads=12, 
    intermediate_size=2048
)

# 2. Fetch and Load Phase 1 Pre-trained Weights
print("Fetching Phase 1 Hybrid Encoder Checkpoint Weights from Hub...")
try:
    model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO_HYBRID, filename="model.safetensors", token=HF_TOKEN)
    state_dict = load_file(model_weight_path)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Loaded Phase 1 weights successfully into Hybrid Encoder Causal LM.")
except Exception as e:
    print(f"Note on weight loading: {e}. Initializing model structure for execution.")

# 3. Freeze Base Weights & Inject LoRA Adapters
print("Injecting LoRA Adapters...")
for param in model.parameters():
    param.requires_grad = False

# Unfreeze LM head
for param in model.lm_head.parameters():
    param.requires_grad = True

# Wrap attention and MLP projections
for layer in model.model.layers:
    for target_attr in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        base_linear = getattr(layer.self_attn, target_attr)
        setattr(layer.self_attn, target_attr, LoRAWrappedLinear(base_linear, rank=CFG["lora_rank"], alpha=CFG["lora_alpha"]))
        
    for target_attr in ["gate_proj", "up_proj", "down_proj"]:
        base_linear = getattr(layer.mlp, target_attr)
        setattr(layer.mlp, target_attr, LoRAWrappedLinear(base_linear, rank=CFG["lora_rank"], alpha=CFG["lora_alpha"]))

# Ensure trainable parameters are FP32
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nPhase 6 Hybrid Encoder LoRA Readiness Summary:")
print(f"  - Total Parameters: {total_params:,}")
print(f"  - Trainable Parameters (LoRA + Head): {trainable_params:,}")
print(f"  - Frozen Base Parameters: {total_params - trainable_params:,}")

Fetching Phase 1 Hybrid Encoder Checkpoint Weights from Hub...


model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded Phase 1 weights successfully into Hybrid Encoder Causal LM.
Injecting LoRA Adapters...

Phase 6 Hybrid Encoder LoRA Readiness Summary:
  - Total Parameters: 93,076,992
  - Trainable Parameters (LoRA + Head): 25,976,832
  - Frozen Base Parameters: 67,100,160


In [6]:
CHECKPOINT_DIR = "./checkpoints_phase6_hybrid_lora"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_hybrid_lora_summ.pt")

epochs = CFG["epochs"]
lr = CFG["lr"]

trainable_param_list = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_param_list, lr=lr, weight_decay=0.01)

best_val_loss = float("inf")

print(f"\nCommencing Abstractive Legal Summarization Training ({epochs} Epochs)...")

for epoch in range(epochs):
    # --------------------------------------------------------------------------
    # 1. Training Phase
    # --------------------------------------------------------------------------
    model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs["loss"]
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Epoch {epoch+1}/{epochs} | Step {step+1}/{len(train_loader)} | Train Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --------------------------------------------------------------------------
    # 2. Validation Phase
    # --------------------------------------------------------------------------
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs["loss"].item()

    avg_val_loss = total_val_loss / len(dev_loader)
    print(f"\nEpoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f}")

    # --------------------------------------------------------------------------
    # 3. Checkpoint Saving
    # --------------------------------------------------------------------------
    if avg_val_loss < best_val_loss or epoch == 0:
        best_val_loss = avg_val_loss
        
        trainable_state_dict = {
            k: v.cpu().clone() for k, v in model.state_dict().items() if v.requires_grad
        }
        
        checkpoint_data = {
            "epoch": epoch + 1,
            "best_val_loss": best_val_loss,
            "trainable_state_dict": trainable_state_dict
        }
        
        torch.save(checkpoint_data, BEST_CHECKPOINT_PATH)
        print(f"  [+] Saved New Best Checkpoint to '{BEST_CHECKPOINT_PATH}' (Val Loss: {best_val_loss:.4f})")
        
    print("-" * 75)


Commencing Abstractive Legal Summarization Training (3 Epochs)...
Epoch 1/3 | Step 20/3164 | Train Loss: 65.7325
Epoch 1/3 | Step 40/3164 | Train Loss: 47.4160
Epoch 1/3 | Step 60/3164 | Train Loss: 18.6623
Epoch 1/3 | Step 80/3164 | Train Loss: 13.5968
Epoch 1/3 | Step 100/3164 | Train Loss: 11.2160
Epoch 1/3 | Step 120/3164 | Train Loss: 9.3506
Epoch 1/3 | Step 140/3164 | Train Loss: 8.4374
Epoch 1/3 | Step 160/3164 | Train Loss: 8.6313
Epoch 1/3 | Step 180/3164 | Train Loss: 6.4791
Epoch 1/3 | Step 200/3164 | Train Loss: 7.9605
Epoch 1/3 | Step 220/3164 | Train Loss: 6.0060
Epoch 1/3 | Step 240/3164 | Train Loss: 6.7227
Epoch 1/3 | Step 260/3164 | Train Loss: 6.3027
Epoch 1/3 | Step 280/3164 | Train Loss: 5.4492
Epoch 1/3 | Step 300/3164 | Train Loss: 6.1083
Epoch 1/3 | Step 320/3164 | Train Loss: 5.6096
Epoch 1/3 | Step 340/3164 | Train Loss: 5.5890
Epoch 1/3 | Step 360/3164 | Train Loss: 4.8479
Epoch 1/3 | Step 380/3164 | Train Loss: 5.3614
Epoch 1/3 | Step 400/3164 | Train Loss:

In [7]:
@torch.no_grad()
def generate_summary(model, tokenizer, prompt_text, max_new_tokens=256, temperature=0.7, top_p=0.9, device="cuda"):
    """
    Autoregressive text generation for legal document summarization.
    """
    model.eval()
    input_ids = tokenizer.encode(prompt_text, truncation=True, max_length=CFG["max_doc_len"], return_tensors="pt").to(device)
    
    generated = input_ids.clone()
    
    for _ in range(max_new_tokens):
        outputs = model(input_ids=generated, attention_mask=torch.ones_like(generated))
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        next_token_logits = logits[:, -1, :] / max(temperature, 1e-5)
        
        # Apply top-p (nucleus) filtering
        sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        
        indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
        next_token_logits[indices_to_remove] = float("-inf")
        
        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        generated = torch.cat([generated, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
            
    prompt_len = input_ids.shape[1]
    gen_tokens = generated[0, prompt_len:].cpu().tolist()
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

print("Autoregressive generation helper configured successfully.")

Autoregressive generation helper configured successfully.


In [8]:
print("Reloading Best Model Checkpoint for Phase 6 Submission Generation...")

if os.path.exists(BEST_CHECKPOINT_PATH):
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print(f"Successfully reloaded checkpoint from Epoch {checkpoint['epoch']} (Best Val Loss: {checkpoint['best_val_loss']:.4f})\n")

test_dataset = LegalSummarizationDataset(test_data, tokenizer, max_doc_len=CFG["max_doc_len"], is_test=True)

model.eval()
predictions = []

print(f"Generating Abstractive Summaries for {len(test_dataset)} Test Documents...")

for idx, sample in enumerate(test_dataset):
    doc_id = sample["id"]
    prompt_text = sample["prompt_text"]
    
    generated_text = generate_summary(
        model=model,
        tokenizer=tokenizer,
        prompt_text=prompt_text,
        max_new_tokens=CFG["gen_max_new_tokens"],
        device=device
    )
    
    if not generated_text:
        generated_text = "The legal document provides rulings and judicial findings on the case."
        
    predictions.append({
        "id": doc_id,
        "summary": generated_text
    })
    
    if (idx + 1) % CFG["log_interval"] == 0 or (idx + 1) == len(test_dataset):
        print(f"Generated {idx+1}/{len(test_dataset)} summaries.")

# Serialization and zip packaging
output_dir = "./phase6_hybrid_lora_summ"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nPhase 6 Hybrid Encoder LoRA Summarization Complete! Generated {len(predictions)} summaries.")
print(f"Submission saved to: {sub_zip_path}")

if DEBUG:
    print("\n[DEBUG] Sample Abstractive Summary Output:")
    print(json.dumps(predictions[:2], indent=2))

Reloading Best Model Checkpoint for Phase 6 Submission Generation...
Successfully reloaded checkpoint from Epoch 3 (Best Val Loss: 0.5046)

Generating Abstractive Summaries for 100 Test Documents...
Generated 20/100 summaries.
Generated 40/100 summaries.
Generated 60/100 summaries.
Generated 80/100 summaries.
Generated 100/100 summaries.

Phase 6 Hybrid Encoder LoRA Summarization Complete! Generated 100 summaries.
Submission saved to: ./phase6_hybrid_lora_summ/submission.zip
